In [1]:
# ====================================================
# Imports
# ====================================================
import os
import gc
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoModel

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.base import clone

import lightgbm as lgb
import xgboost as xgb

# ====================================================
# Config
# ====================================================
class CFG:
    DATA_PATH = '/kaggle/input/csiro-biomass/'
    train_path = os.path.join(DATA_PATH, 'train.csv')
    test_path = os.path.join(DATA_PATH, 'test.csv')
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seeds = [46, 1337, 2024]
    n_fold = 5
    
    img_size = 518
    batch_size = 12
    
    target_cols = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'Dry_Total_g','GDM_g']
    target_weights = [0.1, 0.1, 0.1, 0.2, 0.5]
    
    # Paths to DINO and SSL model checkpoints
    dinov2_path = '/kaggle/input/dinov2/pytorch/large/1'
    ssl_path = '/kaggle/input/resnet18backbone/pytorch/default/1/ssl_backbone_best.pth'  # replace with your path

# ====================================================
# Utilities
# ====================================================
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def weighted_r2_metric(y_true, y_pred):
    weights = np.array(CFG.target_weights)
    if y_true.ndim == 1: y_true = y_true.reshape(-1,1)
    if y_pred.ndim == 1: y_pred = y_pred.reshape(-1,1)
    
    ss_res = np.sum(weights * np.sum((y_true - y_pred)**2, axis=0))
    global_mean = np.average(np.mean(y_true, axis=0), weights=weights)
    ss_tot = np.sum(weights * np.sum((y_true - global_mean)**2, axis=0))
    return 1 - (ss_res / (ss_tot + 1e-8))

def post_process_biomass(df_preds):
    # Physical constraints
    Y = df_preds[['Dry_Green_g','Dry_Clover_g','Dry_Dead_g','GDM_g','Dry_Total_g']].values.T
    C = np.array([[1,1,0,-1,0],[0,0,1,1,-1]])
    try:
        inv_CCt = np.linalg.inv(C @ C.T)
    except:
        inv_CCt = np.linalg.pinv(C @ C.T)
    P = np.eye(5) - C.T @ inv_CCt @ C
    Y_reconciled = (P @ Y).T.clip(min=0)
    df_preds[['Dry_Green_g','Dry_Clover_g','Dry_Dead_g','GDM_g','Dry_Total_g']] = Y_reconciled
    df_preds['GDM_g'] = df_preds['Dry_Green_g'] + df_preds['Dry_Clover_g']
    df_preds['Dry_Total_g'] = df_preds['GDM_g'] + df_preds['Dry_Dead_g']
    return df_preds

FEATURE_DIM = 128    # Must match the training

class SSLResNet(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM):
        super().__init__()
        backbone = resnet18()
        backbone.fc = nn.Identity()  # Remove original classification head
        self.backbone = backbone
        # Projection head (as used in training)
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128)  # <- match your checkpoint
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.projection(h)
        z = F.normalize(z, dim=1)
        return z
# ====================================================
# Dataset
# ====================================================
class BiomassDataset(Dataset):
    def __init__(self, df, transform=None):
        self.image_paths = df['image_path'].values
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = os.path.join(CFG.DATA_PATH, self.image_paths[idx])
        try:
            image = Image.open(path).convert("RGB")
        except:
            image = Image.new("RGB", (256,256))
        if self.transform:
            image = self.transform(image)
        return image

# ====================================================
# Feature extraction (DINO + SSL)
# ====================================================
def extract_features(df, model, ssl_model=None):
    mean = [0.485,0.456,0.406]
    std = [0.229,0.224,0.225]
    
    transforms_list = [
        T.Compose([T.Resize(CFG.img_size), T.CenterCrop(CFG.img_size), T.ToTensor(), T.Normalize(mean,std)]),
        T.Compose([T.Resize(CFG.img_size), T.CenterCrop(CFG.img_size), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(mean,std)]),
        T.Compose([T.Resize(CFG.img_size), T.CenterCrop(CFG.img_size), T.RandomVerticalFlip(p=1.0), T.ToTensor(), T.Normalize(mean,std)])
    ]
    
    feats_tta = []
    
    for i, transform in enumerate(transforms_list):
        ds = BiomassDataset(df, transform=transform)
        loader = DataLoader(
                ds,
                batch_size=CFG.batch_size,
                shuffle=False,
                num_workers=4,
                pin_memory=True,
                persistent_workers=True
            )
        model.eval()
        ssl_features_list = []
        din_features_list = []
        with torch.no_grad():
            for imgs in tqdm(loader, desc=f"TTA {i+1}"):
                imgs = imgs.to(CFG.device)
                
                # DINO features
                if hasattr(model, 'forward_features'):
                    out = model.forward_features(imgs)
                    cls_token = out[:,0,:]
                    patch_tokens = out[:,1:,:]
                    avg_pool = patch_tokens.mean(dim=1)
                    max_pool = patch_tokens.max(dim=1)[0]
                    din_feat = torch.cat([cls_token, avg_pool, max_pool], dim=1)
                else:
                    outputs = model(imgs)
                    last_hidden = outputs.last_hidden_state  # (B, L, D)
                    cls_token = last_hidden[:, 0, :]
                    patch_tokens = last_hidden[:, 1:, :]
                    avg_pool = patch_tokens.mean(dim=1)
                    max_pool = patch_tokens.max(dim=1)[0]
                    din_feat = torch.cat([cls_token, avg_pool, max_pool], dim=1)
                
                din_features_list.append(din_feat.cpu().numpy())
                
                # SSL backbone features
                if ssl_model is not None:
                    ssl_out = ssl_model(imgs)
                    if ssl_out.dim() == 3:  # transformer output
                        cls_token = ssl_out[:,0,:]
                        patch_tokens = ssl_out[:,1:,:]
                        avg_pool = patch_tokens.mean(dim=1)
                        max_pool = patch_tokens.max(dim=1)[0]
                        ssl_feat = torch.cat([cls_token, avg_pool, max_pool], dim=1)
                    else:
                        ssl_feat = ssl_out
                    ssl_features_list.append(ssl_feat.cpu().numpy())
        
        # Combine TTA
        din_feats = np.concatenate(din_features_list, axis=0)
        if ssl_model is not None:
            ssl_feats = np.concatenate(ssl_features_list, axis=0)
            combined = np.concatenate([din_feats, ssl_feats], axis=1)
        else:
            combined = din_feats
        
        feats_tta.append(combined)
    
    final_feats = np.mean(feats_tta, axis=0)
    feat_cols = [f"feat_{i}" for i in range(final_feats.shape[1])]
    return pd.DataFrame(final_feats, columns=feat_cols)

# ====================================================
# Stacking Trainer
# ====================================================
class StackingTrainer:
    def __init__(self, n_folds=CFG.n_fold, seed=46):
        self.n_folds = n_folds
        self.seed = seed
    
    def get_base_models(self):
        lgbm = lgb.LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                random_state=self.seed,
                n_jobs=-1,
                verbose=-1
            )
        xgb_model = xgb.XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,             
                subsample=0.8,
                colsample_bytree=0.8,    
                reg_alpha=1.0,
                reg_lambda=1.0,
                objective='reg:squarederror',
                tree_method='hist',
                device='cuda' if torch.cuda.is_available() else 'cpu',
                random_state=self.seed
            )
        ridge = Ridge(alpha=1.0, random_state=self.seed)
        return {
            'lgbm': MultiOutputRegressor(lgbm),
            'xgb': MultiOutputRegressor(xgb_model),
            'ridge': MultiOutputRegressor(ridge)
        }
    
    def train_predict(self, X, y, X_test):
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.seed)
        base_models = self.get_base_models()
        oof_preds = {name: np.zeros(y.shape) for name in base_models}
        test_preds = {name: np.zeros((X_test.shape[0], y.shape[1])) for name in base_models}
        
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X,y)):
            X_tr, X_val = X[tr_idx], X[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            
            for name, model in base_models.items():
                model_fold = clone(model)
                model_fold.fit(X_tr, y_tr)
                oof_preds[name][val_idx] = np.clip(model_fold.predict(X_val),0,None)
                test_preds[name] += np.clip(model_fold.predict(X_test),0,None)/self.n_folds
        
        return oof_preds, test_preds
    
    def fit_stacking(self, oof_preds, y_true, test_preds_dict):
        final_oof = np.zeros_like(y_true)
        final_test = np.zeros_like(list(test_preds_dict.values())[0])
        for i in range(y_true.shape[1]):
            X_meta = np.column_stack([oof_preds['lgbm'][:,i], oof_preds['xgb'][:,i], oof_preds['ridge'][:,i]])
            X_meta_test = np.column_stack([test_preds_dict['lgbm'][:,i], test_preds_dict['xgb'][:,i], test_preds_dict['ridge'][:,i]])
            meta_model = Ridge(alpha=10.0, random_state=self.seed)
            meta_model.fit(X_meta, y_true[:,i])
            final_oof[:,i] = np.clip(meta_model.predict(X_meta),0,None)
            final_test[:,i] = np.clip(meta_model.predict(X_meta_test),0,None)
        return final_oof, final_test

# ====================================================
# Main
# ====================================================
def main():
    train_df = pd.read_csv(CFG.train_path)
    test_df = pd.read_csv(CFG.test_path)
    
    train_wide = train_df.pivot(index='image_path', columns='target_name', values='target').reset_index()
    train_wide = train_wide[['image_path'] + CFG.target_cols]
    all_seed_preds = []
    for seed in CFG.seeds:
        print(f"\n Seed {seed}")
        seed_everything(seed)
        
        # Load models
        dino_model = AutoModel.from_pretrained(
        '/kaggle/input/dinov2/pytorch/large/1',  # folder containing the .bin & config.json
        local_files_only=True
        ).to(CFG.device).eval()

        ssl_model = SSLResNet(feature_dim=FEATURE_DIM).to(CFG.device)
        
        state_dict = torch.load(CFG.ssl_path, map_location=CFG.device)
        
        # Handle possible 'module.' prefix
        new_sd = {}
        for k, v in state_dict.items():
            kk = k
            if k.startswith("module."):
                kk = k[len("module."):]
            new_sd[kk] = v
        
        ssl_model.load_state_dict(new_sd, strict=False)
        ssl_model.eval()
        
        # Extract features
    
        X_test = extract_features(test_df, dino_model, ssl_model)
        
        # Scaling
        scaler = RobustScaler()
        X_train= extract_features(train_wide, dino_model, ssl_model) 
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Targets
        Y_train_log = np.log1p(train_wide[CFG.target_cols].values)
        
        # Stacking
        trainer = StackingTrainer(n_folds=CFG.n_fold, seed=seed)
        oof_preds_log, test_preds_log = trainer.train_predict(X_train_scaled, Y_train_log, X_test_scaled)
        final_oof_log, final_test_log = trainer.fit_stacking(oof_preds_log, Y_train_log, test_preds_log)

        # ... inside the seed loop ...
        final_oof_log, final_test_log = trainer.fit_stacking(oof_preds_log, Y_train_log, test_preds_log)
        
        # --- ADD THESE LINES TO CALCULATE R2 ---
        # We calculate it in log space because that's what the model was optimized for
        seed_r2 = weighted_r2_metric(Y_train_log, final_oof_log)
        print(f"Seed {seed} | Weighted R2 (Log Space): {seed_r2:.4f}")
        
        # Optional: Calculate for each target to see which one is failing
        for i, col in enumerate(CFG.target_cols):
            from sklearn.metrics import r2_score
            col_r2 = r2_score(Y_train_log[:, i], final_oof_log[:, i])
            print(f"   -> {col}: {col_r2:.4f}")
        # ---------------------------------------

        
        # Back to original space
        final_test = np.expm1(final_test_log)
        all_seed_preds.append(final_test)
        
        del dino_model, ssl_model
        gc.collect()
        torch.cuda.empty_cache()
    
    # Average over seeds
    final_preds = np.mean(all_seed_preds, axis=0)
    
    # Post-process
    df_sub = test_df.copy()
    df_sub[CFG.target_cols] = final_preds
    df_sub = post_process_biomass(df_sub)
    
    df_unique_paths = df_sub.groupby('image_path', as_index=False)[CFG.target_cols].mean()
    
    submission_target_order = ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','Dry_Total_g','GDM_g']
    
    submission_rows = []
    for _, row in df_unique_paths.iterrows():  # loop once per unique image
        # Get the base filename without folders or extension
        # e.g., 'images/test/123.jpg' -> '123'
        base_id = os.path.basename(row['image_path']).split('.')[0]
        
        for col in CFG.target_cols:
            submission_rows.append({
                'sample_id': f"{base_id}__{col}", 
                'target': max(0, float(row[col])) # Ensure no negative values
            })
    
    submission = pd.DataFrame(submission_rows)
    print(submission.head(10))
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("✅ Submission saved successfully!")

if __name__=='__main__':
    main()

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():



 Seed 46


2026-01-08 17:40:50.433268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767894050.621458      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767894050.674041      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767894051.111988      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767894051.112029      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767894051.112031      24 computation_placer.cc:177] computation placer alr

Seed 46 | Weighted R2 (Log Space): 0.8653
   -> Dry_Green_g: 0.8209
   -> Dry_Dead_g: 0.6533
   -> Dry_Clover_g: 0.7841
   -> Dry_Total_g: 0.7509
   -> GDM_g: 0.7954

 Seed 1337


TTA 3: 100%|██████████| 30/30 [01:02<00:00,  2.07s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Seed 1337 | Weighted R2 (Log Space): 0.8666
   -> Dry_Green_g: 0.8163
   -> Dry_Dead_g: 0.6403
   -> Dry_Clover_g: 0.7980
   -> Dry_Total_g: 0.7469
   -> GDM_g: 0.8026

 Seed 2024


TTA 3: 100%|██████████| 30/30 [01:01<00:00,  2.07s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Seed 2024 | Weighted R2 (Log Space): 0.8628
   -> Dry_Green_g: 0.8098
   -> Dry_Dead_g: 0.6463
   -> Dry_Clover_g: 0.7811
   -> Dry_Total_g: 0.7331
   -> GDM_g: 0.7990
                    sample_id     target
0   ID1001187975__Dry_Green_g  34.436819
1    ID1001187975__Dry_Dead_g  34.271627
2  ID1001187975__Dry_Clover_g   3.993701
3   ID1001187975__Dry_Total_g  72.702148
4         ID1001187975__GDM_g  38.430521
✅ Submission saved successfully!
